# LightGBM Meal Ranker Training - Google Colab Version

This notebook trains the LightGBM meal recommendation model using the Food.com dataset.

## Setup Instructions

1. Upload this notebook to Google Colab
2. Upload your Kaggle API credentials (`kaggle.json`) when prompted
3. Run all cells in order
4. Download the generated model files at the end

## Expected Outputs

- `meal_ranker.txt` - Trained LightGBM model (download this)
- `feature_schema.json` - Feature metadata (download this)
- `training_metrics.json` - Model performance metrics (download this)

## Step 1: Install Dependencies

In [ ]:
!pip install lightgbm pandas numpy scikit-learn kaggle -q

## Step 2: Setup Kaggle API Credentials

Upload your `kaggle.json` file when prompted.

In [ ]:
from google.colab import files
import os

# Upload kaggle.json
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle credentials configured successfully!")

## Step 3: Download Food.com Dataset

In [ ]:
!mkdir -p data
!kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -p data --unzip

print("\nDataset downloaded successfully!")
!ls -lh data/

## Step 4: Data Preparation

Load and clean the recipes dataset.

In [ ]:
import pandas as pd
import numpy as np
import ast

# Load recipes
print("Loading recipes...")
df = pd.read_csv('data/RAW_recipes.csv')
print(f"Loaded {len(df)} recipes")

# Parse list columns
print("Parsing columns...")
df['tags'] = df['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['nutrition'] = df['nutrition'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['ingredients'] = df['ingredients'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Extract nutrition values
print("Extracting nutrition values...")
df['calories'] = df['nutrition'].apply(lambda x: float(x[0]) if len(x) > 0 else np.nan)
df['total_fat_g'] = df['nutrition'].apply(lambda x: float(x[1]) if len(x) > 1 else np.nan)
df['protein_g'] = df['nutrition'].apply(lambda x: float(x[4]) if len(x) > 4 else np.nan)
df['carbs_g'] = df['nutrition'].apply(lambda x: float(x[6]) if len(x) > 6 else np.nan)

# Remove invalid rows
df = df.dropna(subset=['calories', 'total_fat_g', 'protein_g', 'carbs_g'])
df = df[(df['calories'] >= 0) & (df['total_fat_g'] >= 0) & (df['protein_g'] >= 0) & (df['carbs_g'] >= 0)]

print(f"Valid recipes: {len(df)}")
df.head()

## Step 5: Infer Cuisine Types and Meal Slots

In [ ]:
def infer_cuisine_type(tags, name):
    """Infer cuisine from tags and name."""
    cuisine_keywords = {
        'italian': ['italian', 'pasta', 'pizza', 'risotto'],
        'mexican': ['mexican', 'taco', 'burrito', 'enchilada'],
        'chinese': ['chinese', 'stir-fry', 'wok'],
        'indian': ['indian', 'curry', 'tandoori', 'masala'],
        'japanese': ['japanese', 'sushi', 'teriyaki', 'ramen'],
        'french': ['french', 'provence', 'crepe'],
        'thai': ['thai', 'pad-thai'],
        'greek': ['greek', 'mediterranean', 'gyro'],
    }
    
    tags_lower = [str(tag).lower() for tag in tags]
    name_lower = str(name).lower()
    search_text = ' '.join(tags_lower) + ' ' + name_lower
    
    for cuisine, keywords in cuisine_keywords.items():
        if any(kw in search_text for kw in keywords):
            return cuisine
    return 'american'

def infer_meal_slot(tags, name):
    """Classify into meal slot."""
    tags_lower = [str(tag).lower() for tag in tags]
    name_lower = str(name).lower()
    search_text = ' '.join(tags_lower) + ' ' + name_lower
    
    if any(kw in search_text for kw in ['breakfast', 'brunch', 'pancake', 'waffle', 'oatmeal']):
        return 'breakfast'
    if any(kw in search_text for kw in ['snack', 'appetizer', 'finger-food', 'candy']):
        return 'snack'
    if any(kw in search_text for kw in ['lunch', 'sandwich', 'wrap']):
        return 'lunch'
    return 'dinner'

print("Inferring cuisine types and meal slots...")
df['cuisine_type'] = df.apply(lambda row: infer_cuisine_type(row['tags'], row['name']), axis=1)
df['meal_slot'] = df.apply(lambda row: infer_meal_slot(row['tags'], row['name']), axis=1)

print("\nCuisine distribution:")
print(df['cuisine_type'].value_counts())
print("\nMeal slot distribution:")
print(df['meal_slot'].value_counts())

## Step 6: Simulate User Profiles

In [ ]:
import random

def simulate_user_profiles(n=5000):  # Reduced for Colab
    """Generate synthetic user profiles."""
    np.random.seed(42)
    random.seed(42)
    
    activity_levels = ['sedentary', 'lightly_active', 'moderately_active', 'very_active', 'extra_active']
    goals = ['weight_loss', 'maintenance', 'muscle_gain']
    
    profiles = []
    for i in range(n):
        profiles.append({
            'user_id': f'user_{i}',
            'age': int(np.clip(np.random.normal(35, 15), 18, 80)),
            'weight_kg': float(np.clip(np.random.normal(75, 20), 50, 150)),
            'height_cm': float(np.clip(np.random.normal(170, 12), 150, 200)),
            'activity_level': random.choice(activity_levels),
            'goal': random.choice(goals)
        })
    
    return pd.DataFrame(profiles)

print("Simulating user profiles...")
users = simulate_user_profiles()
print(f"Generated {len(users)} user profiles")
users.head()

## Step 7: Feature Engineering Functions

In [ ]:
def compute_tdee(profile):
    """Calculate TDEE using Mifflin-St Jeor."""
    bmr = 10 * profile['weight_kg'] + 6.25 * profile['height_cm'] - 5 * profile['age'] - 78
    multipliers = {'sedentary': 1.2, 'lightly_active': 1.375, 'moderately_active': 1.55,
                   'very_active': 1.725, 'extra_active': 1.9}
    return bmr * multipliers.get(profile['activity_level'], 1.55)

def compute_macro_targets(profile, tdee):
    """Calculate daily macro targets."""
    goal = profile['goal']
    weight_kg = profile['weight_kg']
    
    if goal == 'weight_loss':
        target_cal = tdee - 500
        protein_per_kg, fat_pct = 2.0, 0.25
    elif goal == 'muscle_gain':
        target_cal = tdee + 300
        protein_per_kg, fat_pct = 2.2, 0.25
    else:
        target_cal = tdee
        protein_per_kg, fat_pct = 1.6, 0.30
    
    protein_g = weight_kg * protein_per_kg
    fat_g = (target_cal * fat_pct) / 9
    carbs_g = (target_cal - protein_g * 4 - fat_g * 9) / 4
    
    return {
        'calories': target_cal,
        'protein_g': protein_g,
        'carbs_g': max(0, carbs_g),
        'fat_g': fat_g
    }

def simulate_remaining_macros(targets):
    """Simulate remaining macros."""
    scenario = random.choice(['early', 'mid', 'late'])
    if scenario == 'early':
        consumption_pct = np.random.beta(2, 8)
    elif scenario == 'mid':
        consumption_pct = np.random.beta(5, 5)
    else:
        consumption_pct = np.random.beta(8, 2)
    
    remaining = {}
    for key in targets:
        remaining_pct = 1 - (consumption_pct * np.random.uniform(0.9, 1.1))
        remaining[key] = max(0, targets[key] * np.clip(remaining_pct, 0, 1))
    return remaining

def engineer_features(user_context, meal, remaining):
    """Create 7-feature vector."""
    # Macro deltas
    cal_delta = abs(meal['calories'] - remaining['calories'])
    prot_delta = abs(meal['protein_g'] - remaining['protein_g'])
    carb_delta = abs(meal['carbs_g'] - remaining['carbs_g'])
    fat_delta = abs(meal['fat_g'] - remaining['fat_g'])
    
    # Cuisine match
    cuisine_match = 1 if (user_context.get('cuisine_preference') and 
                          meal['cuisine_type'].lower() == user_context['cuisine_preference'].lower()) else 0
    
    # Meal slot match
    slot_match = 1 if meal['meal_slot'].lower() == user_context['requested_slot'].lower() else 0
    
    # Days since last eaten (simplified)
    days_since = 999  # Assume never eaten for training
    
    return [cal_delta, prot_delta, carb_delta, fat_delta, cuisine_match, slot_match, days_since]

def compute_synthetic_rating(features):
    """Generate synthetic rating based on features."""
    rating = 3.0
    rating -= min(features[0] / 500, 1.0) * 1.5  # Cal penalty
    rating -= min(features[1] / 50, 1.0) * 0.5   # Protein penalty
    rating += 0.5 if features[4] == 1 else 0     # Cuisine boost
    rating += 1.0 if features[5] == 1 else -1.0  # Slot boost/penalty
    rating += 0.2 if features[6] > 14 else 0     # Recency boost
    rating = max(1.0, min(5.0, rating + np.random.normal(0, 0.2)))
    return rating

print("Feature engineering functions defined!")

## Step 8: Create Training Pairs

This may take 10-20 minutes depending on dataset size.

In [ ]:
print("Creating training pairs...")
print(f"Users: {len(users)}, Recipes: {len(df)}")

X = []
y = []

cuisines = ['american', 'italian', 'mexican', 'chinese', 'indian', None]
meal_slots = ['breakfast', 'lunch', 'dinner', 'snack']
contexts_per_user = 5  # Reduced for Colab

for idx, user_row in users.iterrows():
    if idx % 500 == 0:
        print(f"  Processed {idx}/{len(users)} users... ({len(X):,} pairs)")
    
    user_profile = user_row.to_dict()
    tdee = compute_tdee(user_profile)
    targets = compute_macro_targets(user_profile, tdee)
    
    for _ in range(contexts_per_user):
        remaining = simulate_remaining_macros(targets)
        cuisine_pref = random.choice(cuisines)
        requested_slot = random.choice(meal_slots)
        
        user_context = {
            'cuisine_preference': cuisine_pref,
            'requested_slot': requested_slot
        }
        
        # Sample 50 random recipes per context to reduce memory
        sample_recipes = df.sample(n=min(50, len(df)))
        
        for _, recipe_row in sample_recipes.iterrows():
            meal = {
                'calories': recipe_row['calories'],
                'protein_g': recipe_row['protein_g'],
                'carbs_g': recipe_row['carbs_g'],
                'fat_g': recipe_row['total_fat_g'],
                'cuisine_type': recipe_row['cuisine_type'],
                'meal_slot': recipe_row['meal_slot']
            }
            
            features = engineer_features(user_context, meal, remaining)
            rating = compute_synthetic_rating(features)
            
            X.append(features)
            y.append(rating)

X = np.array(X)
y = np.array(y)

print(f"\n✓ Generated {len(X):,} training pairs")
print(f"  Feature shape: {X.shape}")
print(f"  Label range: {y.min():.2f} to {y.max():.2f}")

## Step 9: Train LightGBM Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, ndcg_score
import lightgbm as lgb

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

print("\nTraining LightGBM model...")
model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)
print("✓ Training complete!")

## Step 10: Evaluate Model

In [ ]:
print("Evaluating model...")

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"\nMean Absolute Error: {mae:.4f}")

# NDCG@5
group_size = 100
n_groups = len(y_test) // group_size
ndcg_scores = []
precision_scores = []

for i in range(n_groups):
    start = i * group_size
    end = start + group_size
    
    y_true_group = y_test[start:end]
    y_pred_group = y_pred[start:end]
    
    try:
        ndcg = ndcg_score([y_true_group], [y_pred_group], k=5)
        ndcg_scores.append(ndcg)
    except:
        pass
    
    top5_indices = np.argsort(y_pred_group)[-5:]
    precision = (y_true_group[top5_indices] >= 4.0).sum() / 5.0
    precision_scores.append(precision)

avg_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0
avg_precision = np.mean(precision_scores)

print(f"NDCG@5: {avg_ndcg:.4f} (target: >= 0.3)")
print(f"Precision@5: {avg_precision:.4f} (target: >= 0.6)")

if avg_ndcg < 0.3:
    print("\n⚠️  Warning: NDCG@5 is below target. Consider training longer or adjusting features.")
else:
    print("\n✓ Model meets performance targets!")

metrics = {
    'mae': float(mae),
    'ndcg_5': float(avg_ndcg),
    'precision_5': float(avg_precision),
    'n_training_samples': len(X_train),
    'n_test_samples': len(X_test)
}

## Step 11: Save Model and Metadata

In [ ]:
import json

# Save model
print("Saving model...")
model.booster_.save_model('meal_ranker.txt')
print("✓ Model saved to meal_ranker.txt")

# Save feature schema
feature_schema = {
    'version': '1.0',
    'feature_count': 7,
    'features': [
        {'name': 'macro_delta_calories', 'type': 'float', 'min': 0, 'max': 2000},
        {'name': 'macro_delta_protein', 'type': 'float', 'min': 0, 'max': 200},
        {'name': 'macro_delta_carbs', 'type': 'float', 'min': 0, 'max': 300},
        {'name': 'macro_delta_fat', 'type': 'float', 'min': 0, 'max': 100},
        {'name': 'cuisine_match', 'type': 'int', 'min': 0, 'max': 1},
        {'name': 'meal_slot_match', 'type': 'int', 'min': 0, 'max': 1},
        {'name': 'days_since_last_eaten', 'type': 'int', 'min': 0, 'max': 999}
    ]
}

with open('feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)
print("✓ Feature schema saved to feature_schema.json")

# Save metrics
with open('training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ Metrics saved to training_metrics.json")

print("\n" + "="*60)
print("Training Complete!")
print("="*60)
print("\nGenerated files:")
print("  - meal_ranker.txt (model file)")
print("  - feature_schema.json (feature metadata)")
print("  - training_metrics.json (performance metrics)")
print("\nDownload these files and copy them to your local lib/meal-recommender/ directory.")

## Step 12: Download Files

Download the generated files to your local machine.

In [ ]:
from google.colab import files

print("Downloading files...")
files.download('meal_ranker.txt')
files.download('feature_schema.json')
files.download('training_metrics.json')
print("\n✓ Downloads complete! Copy these files to lib/meal-recommender/ in your project.")